# Day 3: Ingest JSON into Bronze using Auto Loader
Requirement: Ingest JSON data into Bronze using Auto Loader.
Apply add audit columns (load_dt, source).
Create and add descriptions/metadata about enterprise data to make it more discoverable.

In [0]:
import pyspark.sql.functions as F

# Define paths and table names
source_path = "/Volumes/vstone/bronze/raw_volume/json/"
checkpoint_path = "/Volumes/vstone/bronze/raw_volume/checkpoints/bronze_json_autoloader/"
catalog = "vstone"
schema = "bronze"
bronze_table_name = "flight_json_bronze"
bronze_table_full_name = f"{catalog}.{schema}.{bronze_table_name}"

### 1. Read Stream using Auto Loader (`cloudFiles`)

In [0]:
# Configure Auto Loader to read JSON
# Auto Loader uses cloudFiles format
raw_stream_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
  .option("cloudFiles.inferColumnTypes", "true") # Highly recommended for JSON
  .load(source_path)
)

# Add Audit Columns
bronze_stream_df = (raw_stream_df
  .withColumn("load_dt", F.current_timestamp())
  .withColumn("source", F.col("_metadata.file_path"))
)

### 2. Write Stream to Delta Table (Bronze)

In [0]:
# Write stream to Delta Table
query = (bronze_stream_df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True) # Recommended for batch-like execution of streaming jobs
  .toTable(bronze_table_full_name)
)

# Await termination if running interactively, though in a Job it's often run once with availableNow=True
query.awaitTermination()

print(f"Auto Loader ingestion completed for {bronze_table_full_name}")

### 3. Add Metadata and Descriptions

In [0]:
# Add table description
spark.sql(f"COMMENT ON TABLE {bronze_table_full_name} IS 'Bronze layer table for Flight Delays ingested from JSON using Auto Loader. Contains raw data with audit columns.'")

print("Day 3: Auto Loader JSON ingestion completed successfully.")

In [0]:
%sql
select * from vstone.bronze.flight_json_bronze
limit 10;

In [0]:
%sql
select count(*) from vstone.bronze.flight_json_bronze;